# Day 4: Fund Performance Analytics

## Objectives
- Compute all key performance and risk metrics from NAV history.
- Build a fund ranking/scoring model (Fund Scorecard).
- Compare fund returns against benchmark indices (Nifty 50 and Nifty 100).
- Identify the best and worst performing funds per category.
- Organize output files and update the SQLite database.

### Mathematical Formulas Used

1. **Daily Returns**:
   $$R_t = \frac{NAV_t}{NAV_{t-1}} - 1$$
   
2. **Annualised Return**:
   $$\text{Annualised Return} = \prod_{t=1}^{n} (1 + R_t)^{252/n} - 1$$
   
3. **CAGR (1yr, 3yr, 5yr)**:
   $$\text{CAGR} = \left(\frac{NAV_{end}}{NAV_{start}}\right)^{1/n} - 1$$
   where $n$ is the number of years (using exact days / 365.25).
   
4. **Sharpe Ratio**:
   $$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p \times \sqrt{252}}$$
   where $R_f = 6.5\%$ and $\sigma_p$ is daily standard deviation.
   
5. **Sortino Ratio**:
   $$\text{Sortino} = \frac{R_p - R_f}{\sigma_d \times \sqrt{252}}$$
   where $\sigma_d$ is the standard deviation of negative returns.

6. **Alpha & Beta**:
   OLS regression of fund returns on Nifty 100 returns:
   $$R_{\text{fund}} = \alpha + \beta \cdot R_{\text{benchmark}} + \epsilon$$
   $$\text{Annualized Alpha} = \alpha_{\text{daily}} \times 252$$
   $$\text{Beta} = \beta$$

7. **Maximum Drawdown**:
   $$\text{Drawdown} = \frac{NAV_t}{\text{Running Max NAV}} - 1$$
   $$\text{Max Drawdown} = \min(\text{Drawdown})$$

8. **Tracking Error**:
   $$\text{Tracking Error} = \text{std}(R_{\text{fund}} - R_{\text{benchmark}}) \times \sqrt{252}$$

9. **Fund Scorecard (Composite Score 0-100)**:
   $$\text{Score} = 30\% \times (\text{3yr Return Rank}) + 25\% \times (\text{Sharpe Rank}) + 20\% \times (\text{Alpha Rank}) + 15\% \times (\text{Expense Ratio Rank, inverse}) + 10\% \times (\text{Max DD Rank, inverse})$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from pathlib import Path
import sqlite3

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["figure.figsize"] = (12, 6)

BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports" / "day4"
DB_PATH = BASE_DIR / "database" / "bluestock_mf.db"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Cleaned Datasets

In [ ]:
nav_df = pd.read_csv(PROCESSED_DIR / "clean_nav.csv", parse_dates=["date"])
master_df = pd.read_csv(PROCESSED_DIR / "01_fund_master.csv")
bench_df = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

print(f"Loaded {len(nav_df)} NAV records across {len(master_df)} schemes.")

## 2. Compute Daily Returns & Annualized Returns

In [ ]:
nav_pivot = nav_df.pivot(index="date", columns="amfi_code", values="nav").sort_index()
daily_returns = nav_pivot.pct_change()

ann_returns = {}
for code in daily_returns.columns:
    ret_series = daily_returns[code].dropna()
    n = len(ret_series)
    if n > 0:
        ann_ret = (1 + ret_series).prod() ** (252 / n) - 1
        ann_returns[code] = ann_ret
    else:
        ann_returns[code] = np.nan

print("Sample Annualized Returns:")
for code in list(ann_returns.keys())[:5]:
    name = master_df[master_df["amfi_code"] == code]["scheme_name"].values[0]
    print(f"- {name}: {ann_returns[code]:.2%}")

## 3. Calculate CAGR (1yr, 3yr, 5yr proxy)

In [ ]:
latest_date = nav_pivot.index.max()
cagr_results = []

for code in nav_pivot.columns:
    fund_nav = nav_pivot[code].dropna()
    if fund_nav.empty: continue
    
    val_end = fund_nav.loc[latest_date]
    date_end = fund_nav.index[-1]
    
    # 1yr
    target_1yr = date_end - pd.DateOffset(years=1)
    available_1yr = fund_nav.index[fund_nav.index <= target_1yr]
    cagr_1yr = (val_end / fund_nav.loc[available_1yr[-1]]) ** (1 / ((date_end - available_1yr[-1]).days / 365.25)) - 1 if len(available_1yr) > 0 else np.nan
    
    # 3yr
    target_3yr = date_end - pd.DateOffset(years=3)
    available_3yr = fund_nav.index[fund_nav.index <= target_3yr]
    cagr_3yr = (val_end / fund_nav.loc[available_3yr[-1]]) ** (1 / ((date_end - available_3yr[-1]).days / 365.25)) - 1 if len(available_3yr) > 0 else np.nan
    
    # 5yr (proxy over maximum history)
    date_start_5yr = fund_nav.index[0]
    val_start_5yr = fund_nav.iloc[0]
    n_years_5yr = (date_end - date_start_5yr).days / 365.25
    cagr_5yr = (val_end / val_start_5yr) ** (1 / n_years_5yr) - 1
    
    cagr_results.append({
        "amfi_code": code,
        "cagr_1yr": cagr_1yr,
        "cagr_3yr": cagr_3yr,
        "cagr_5yr": cagr_5yr
    })

cagr_df = pd.DataFrame(cagr_results)
cagr_df = cagr_df.merge(master_df[["amfi_code", "scheme_name"]], on="amfi_code")
print(cagr_df[["scheme_name", "cagr_1yr", "cagr_3yr", "cagr_5yr"]].head())

## 4. Compute Sharpe & Sortino Ratios

In [ ]:
rf_annual = 0.065
sharpe_sortino_results = []

for code in daily_returns.columns:
    returns = daily_returns[code].dropna()
    if len(returns) == 0: continue
    
    rp_annual = returns.mean() * 252
    vol_annual = returns.std() * np.sqrt(252)
    
    sharpe = (rp_annual - rf_annual) / vol_annual if vol_annual > 0 else np.nan
    
    # Sortino
    neg_returns = returns[returns < 0]
    downside_std_annual = neg_returns.std() * np.sqrt(252)
    sortino = (rp_annual - rf_annual) / downside_std_annual if downside_std_annual > 0 else np.nan
    
    sharpe_sortino_results.append({
        "amfi_code": code,
        "rp_annualized": rp_annual,
        "vol_annualized": vol_annual,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino
    })

sharpe_sortino_df = pd.DataFrame(sharpe_sortino_results)
sharpe_sortino_df = sharpe_sortino_df.merge(master_df[["amfi_code", "scheme_name"]], on="amfi_code")
print(sharpe_sortino_df[["scheme_name", "sharpe_ratio", "sortino_ratio"]].head())

## 5. Compute Alpha & Beta vs. Nifty 100 Index

In [ ]:
nifty100_df = bench_df[bench_df["index_name"] == "NIFTY100"].sort_values("date").copy()
nifty100_df["benchmark_return"] = nifty100_df["close_value"].pct_change()
nifty100_returns = nifty100_df.set_index("date")["benchmark_return"].dropna()

alpha_beta_results = []

for code in daily_returns.columns:
    fund_ret = daily_returns[code].dropna()
    merged = pd.concat([fund_ret, nifty100_returns], axis=1, join="inner").dropna()
    
    if len(merged) < 5:
        alpha_beta_results.append({"amfi_code": code, "alpha": np.nan, "beta": np.nan})
        continue
        
    slope, intercept, _, _, _ = linregress(merged.iloc[:, 1], merged.iloc[:, 0])
    alpha_beta_results.append({
        "amfi_code": code,
        "alpha": intercept * 252,
        "beta": slope
    })

alpha_beta_df = pd.DataFrame(alpha_beta_results)
alpha_beta_df = alpha_beta_df.merge(master_df[["amfi_code", "scheme_name"]], on="amfi_code")
print(alpha_beta_df[["scheme_name", "alpha", "beta"]].head())

## 6. Compute Maximum Drawdown

In [ ]:
max_dd_results = []

for code in nav_pivot.columns:
    fund_nav = nav_pivot[code].dropna()
    if fund_nav.empty: continue
    
    running_max = fund_nav.cummax()
    drawdown = fund_nav / running_max - 1
    max_dd = drawdown.min()
    
    trough_idx = drawdown.idxmin()
    peak_idx = fund_nav.loc[:trough_idx].idxmax()
    
    max_dd_results.append({
        "amfi_code": code,
        "max_drawdown_pct": max_dd * 100,
        "peak_date": peak_idx.strftime("%Y-%m-%d"),
        "trough_date": trough_idx.strftime("%Y-%m-%d")
    })

max_dd_df = pd.DataFrame(max_dd_results)
max_dd_df = max_dd_df.merge(master_df[["amfi_code", "scheme_name"]], on="amfi_code")
print(max_dd_df[["scheme_name", "max_drawdown_pct", "peak_date", "trough_date"]].head())

## 7. Build Fund Scorecard & Rank Schemes

In [ ]:
scorecard = cagr_df[["amfi_code", "scheme_name", "cagr_3yr"]].copy()
scorecard["cagr_3yr_pct"] = scorecard["cagr_3yr"] * 100
scorecard = scorecard.merge(sharpe_sortino_df[["amfi_code", "sharpe_ratio"]], on="amfi_code")
scorecard = scorecard.merge(alpha_beta_df[["amfi_code", "alpha"]], on="amfi_code")
scorecard = scorecard.merge(max_dd_df[["amfi_code", "max_drawdown_pct"]], on="amfi_code")

expense_map = master_df.set_index("amfi_code")["expense_ratio_pct"].to_dict()
cat_map = master_df.set_index("amfi_code")["category"].to_dict()
scorecard["expense_ratio_pct"] = scorecard["amfi_code"].map(expense_map)
scorecard["category"] = scorecard["amfi_code"].map(cat_map)

scorecard = scorecard.dropna().copy()

# Calculate percentile ranks
scorecard["rank_3yr"] = scorecard["cagr_3yr_pct"].rank(pct=True) * 100
scorecard["rank_sharpe"] = scorecard["sharpe_ratio"].rank(pct=True) * 100
scorecard["rank_alpha"] = scorecard["alpha"].rank(pct=True) * 100
scorecard["rank_expense_inv"] = scorecard["expense_ratio_pct"].rank(ascending=False, pct=True) * 100
scorecard["rank_max_dd_inv"] = scorecard["max_drawdown_pct"].rank(ascending=True, pct=True) * 100

# Weighted Score
scorecard["composite_score"] = (
    0.30 * scorecard["rank_3yr"] +
    0.25 * scorecard["rank_sharpe"] +
    0.20 * scorecard["rank_alpha"] +
    0.15 * scorecard["rank_expense_inv"] +
    0.10 * scorecard["rank_max_dd_inv"]
)

scorecard = scorecard.sort_values("composite_score", ascending=False).reset_index(drop=True)
scorecard["final_rank"] = scorecard.index + 1

print("Top 5 Funds by Scorecard:")
print(scorecard[["final_rank", "scheme_name", "category", "composite_score"]].head())

# Save to reports/day4
scorecard.to_csv(REPORTS_DIR / "fund_scorecard.csv", index=False)

## 8. Benchmark Comparison Visualization

In [ ]:
top_5_codes = scorecard.head(5)["amfi_code"].tolist()
start_date_3yr = latest_date - pd.DateOffset(years=3)

nav_3yr_pivot = nav_pivot[top_5_codes].loc[start_date_3yr:latest_date].ffill()
nav_indexed = nav_3yr_pivot.div(nav_3yr_pivot.iloc[0]) * 100

bench_3yr_pivot = bench_df[bench_df["index_name"].isin(["NIFTY50", "NIFTY100"])].pivot(index="date", columns="index_name", values="close_value")
bench_indexed = bench_3yr_pivot.loc[start_date_3yr:latest_date].ffill().div(bench_3yr_pivot.loc[start_date_3yr:latest_date].ffill().iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 8), dpi=150)
colors = ["#0F766E", "#2563EB", "#7C3AED", "#DC2626", "#F59E0B"]

for i, code in enumerate(top_5_codes):
    name = scorecard[scorecard["amfi_code"] == code]["scheme_name"].values[0]
    ax.plot(nav_indexed.index, nav_indexed[code], label=name.split(" - ")[0], color=colors[i], linewidth=2.5)

ax.plot(bench_indexed.index, bench_indexed["NIFTY50"], label="NIFTY 50", color="#10B981", linestyle="--", linewidth=3)
ax.plot(bench_indexed.index, bench_indexed["NIFTY100"], label="NIFTY 100", color="#4B5563", linestyle="-.", linewidth=3)

ax.set_title("Top 5 Funds vs Benchmarks (3-Year Cumulative Return)", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Indexed Value (Base = 100)", fontsize=12)
ax.legend(loc="upper left", frameon=True, facecolor="white")

# Save chart
plt.savefig(REPORTS_DIR / "benchmark_comparison_chart.png", bbox_inches="tight", dpi=150)
plt.show()

## 9. SQLite Database Verification

In [ ]:
if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT amfi_code, scheme_name, return_3yr_pct, sharpe_ratio, alpha, max_drawdown_pct FROM fact_performance LIMIT 5", conn)
    conn.close()
    print("Updated DB fact_performance values:")
    print(df)
else:
    print(f"Database not found at: {DB_PATH}")